# Calculate OBSO for a given run

In [1]:
import os

# Print current working directory
print(os.getcwd())

# Change to your desired directory
os.chdir('/Users/svasquez/driblab/dl-vision')
print(os.getcwd())

/Users/svasquez/driblab/dl-vision/stats/obso
/Users/svasquez/driblab/dl-vision


In [2]:
import json
import pandas as pd

In [3]:
# Load runs file

match_id = 666385
run_path = f"outputs/{match_id}/{match_id}_runs.jsonl"

def load_jsonl(file_path):
    runs = []
    with open(file_path, 'r') as f:
        for line in f:
            runs.append(json.loads(line))
    return runs

runs = load_jsonl(run_path)
print(f"Loaded {len(runs)} runs")


Loaded 455 runs


## take a random RUn to test

In [4]:
run = runs[375]
run

{'match_id': '666385',
 'run_id': '666385_375',
 'team_id': 3651,
 'player_id': 1157794,
 'ball_carrier_id_start': 1610030,
 'ball_carrier_id_end': 1157794,
 'start_frame': 46125,
 'end_frame': 46161,
 'start_timestamp': 4641.74,
 'end_timestamp': 4645.34,
 'period': 2,
 'runner_info': {'start_position': {'x': 62.0, 'y': 54.56},
  'positions': [{'x': 62.0, 'y': 54.56, 'frame': 46125, 'period': 2}],
  'metrics': {'xR_start': None,
   'xR_end': None,
   'PC_start': None,
   'PC_end': None,
   'xG_start': None,
   'xG_end': None,
   'OBSO_start': None,
   'OBSO_end': None},
  'end_position': {'x': 85.08, 'y': 53.14}},
 'possession': {'start_team': '3651', 'end_team': '3651'},
 'off_ball_teammates': [{'player_id': 38689,
   'start_position': {'x': 52.45, 'y': 23.44},
   'end_position': {'x': 83.35, 'y': 28.88},
   'metrics': {'xR_start': None,
    'xR_end': None,
    'PC_start': None,
    'PC_end': None,
    'xG_start': None,
    'xG_end': None,
    'OBSO_start': None,
    'OBSO_end': None

## For this run, calculate the xR at the start and end of the Run

In [5]:
# 

In [8]:
from stats.xReceiver.xgboost_model import XGBoostModel

model = XGBoostModel(model_path="stats/xReceiver/models/xgb_model_no_pc.pkl")
model.load_data(match_id=666385)

model.xgb_run_processor.load_run(run)


Loading run: {'match_id': '666385', 'run_id': '666385_375', 'team_id': 3651, 'player_id': 1157794, 'ball_carrier_id_start': 1610030, 'ball_carrier_id_end': 1157794, 'start_frame': 46125, 'end_frame': 46161, 'start_timestamp': 4641.74, 'end_timestamp': 4645.34, 'period': 2, 'runner_info': {'start_position': {'x': 62.0, 'y': 54.56}, 'positions': [{'x': 62.0, 'y': 54.56, 'frame': 46125, 'period': 2}], 'metrics': {'xR_start': None, 'xR_end': None, 'PC_start': None, 'PC_end': None, 'xG_start': None, 'xG_end': None, 'OBSO_start': None, 'OBSO_end': None}, 'end_position': {'x': 85.08, 'y': 53.14}}, 'possession': {'start_team': '3651', 'end_team': '3651'}, 'off_ball_teammates': [{'player_id': 38689, 'start_position': {'x': 52.45, 'y': 23.44}, 'end_position': {'x': 83.35, 'y': 28.88}, 'metrics': {'xR_start': None, 'xR_end': None, 'PC_start': None, 'PC_end': None, 'xG_start': None, 'xG_end': None, 'OBSO_start': None, 'OBSO_end': None}}, {'player_id': 1219428, 'start_position': {'x': 34.68, 'y': 3

In [9]:
features = model.xgb_run_processor.get_features(type='start', plyr_id=1157794)

xP_pred, class_pred  = model.predict(features)

print(f"xP_pred: {xP_pred}")
print(f"class_pred: {class_pred}")

{'distance': 25.375965006281042, 'theta1': 18.39956561342975, 'theta2': 165.47385881003052, 'dir_angle': 26.039972374456447, 'giver_coord.x': 39.2, 'giver_coord.y': 43.42, 'receiver_coord.x': 62.0, 'receiver_coord.y': 54.56, 'giver_speed': 8.273451516794372, 'receiver_speed': 5.510898293412006, 'tm_count_5': 0, 'tm_count_10': 5, 'opp_count_2': 0, 'opp_count_5': 1, 'opp_count_10': 2, 'dist_giv_sideline': 39.2, 'dist_rec_sideline': 38.0, 'num_opp2plyr_goal_giv': 0, 'num_opp2plyr_goal_rec': 0, 'dist_giv_goal': 66.47086880732039, 'dist_rec_goal': 47.66249678730647, 'dist_opponent_giv': 3.007457397869504, 'dist_opponent_rec': 3.6487806182339892, 'opp_in_corridor_count': 2, 'dist2offside': 8.379999999999995, 'offside': False}
xP_pred: 0.6758465766906738
class_pred: 1


In [11]:
def get_xR(model,plyr_id, type):
    features = model.xgb_run_processor.get_features(type=type, plyr_id=plyr_id)
    xP_pred, class_pred  = model.predict(features)
    return xP_pred, class_pred


In [17]:
# Predict the xR of the runner

runner_plyr_id = run['player_id']

runner_xR_start, class_pred = get_xR(model, runner_plyr_id, 'start')
runner_xR_end, class_pred = get_xR(model, runner_plyr_id, 'end')

print(f"runner_xR_start: {runner_xR_start:.2f} , runner_xR_end: {runner_xR_end:.2f}")


# Predict the xR of the other off ball players

off_ball_plyrs = run['off_ball_teammates']

for plyr in off_ball_plyrs:
    plyr_xR_start, class_pred = get_xR(model, plyr['player_id'], 'start')
    plyr_xR_end, class_pred = get_xR(model, plyr['player_id'], 'end')
    print(f"plyr: {plyr['player_id']} , plyr_xR_start: {plyr_xR_start:.2f} , plyr_xR_end: {plyr_xR_end:.2f}")







runner_xR_start: 0.68 , runner_xR_end: 0.48
plyr: 38689 , plyr_xR_start: 0.39 , plyr_xR_end: 0.40
plyr: 1219428 , plyr_xR_start: 0.38 , plyr_xR_end: 0.50
plyr: 1329659 , plyr_xR_start: 0.58 , plyr_xR_end: 0.44
plyr: 1263688 , plyr_xR_start: 0.36 , plyr_xR_end: 0.49
plyr: 45067 , plyr_xR_start: 0.57 , plyr_xR_end: 0.45
plyr: 1610030 , plyr_xR_start: 0.49 , plyr_xR_end: 0.42
plyr: 69811 , plyr_xR_start: 0.37 , plyr_xR_end: 0.48
plyr: 239959 , plyr_xR_start: 0.38 , plyr_xR_end: 0.43
plyr: 1159675 , plyr_xR_start: 0.37 , plyr_xR_end: 0.50


/Users/svasquez/driblab/dl-vision/stats/xReceiver/xR_get_features_v2.py:124: RuntimeWarning: invalid value encountered in divide
  D1_norm = D1 / np.linalg.norm(D1)
/Users/svasquez/driblab/dl-vision/stats/xReceiver/xR_get_features_v2.py:126: RuntimeWarning: invalid value encountered in divide
  D2_norm = D2 / np.linalg.norm(D2)
/Users/svasquez/driblab/dl-vision/stats/xReceiver/xR_get_features_v2.py:161: RuntimeWarning: invalid value encountered in double_scalars
  cos_theta = dot_product / magnitude_pass


In [7]:
model.features_cols

['distance',
 'theta1',
 'theta2',
 'dir_angle',
 'giver_coord.x',
 'giver_coord.y',
 'receiver_coord.x',
 'receiver_coord.y',
 'giver_speed',
 'receiver_speed',
 'tm_count_5',
 'tm_count_10',
 'opp_count_2',
 'opp_count_5',
 'opp_count_10',
 'dist_giv_sideline',
 'dist_rec_sideline',
 'num_opp2plyr_goal_giv',
 'num_opp2plyr_goal_rec',
 'dist_giv_goal',
 'dist_rec_goal',
 'dist_opponent_giv',
 'dist_opponent_rec',
 'opp_in_corridor_count',
 'dist2offside',
 'offside']